### Business Objective

The goal of this model is not only to predict default risk, but to support
**transparent, risk-aware credit decisions**.

Predictions are used to:
- Rank applicants by risk
- Assign applicants to risk tiers
- Provide explanations for approvals or rejections

We use the calibrated probability of default (PD) as the primary risk signal,
rather than raw model scores.

In [ ]:
import pandas as pd
import numpy as np
import joblib

model = joblib.load("C:/Users/Pratik/DS/credit-risk-ml/models/logreg_platt.joblib")
X_test, y_test = joblib.load('C:/Users/Pratik/DS/credit-risk-ml/models/test_data.joblib')

y_prob = model.predict_proba(X_test)[:, 1]

In [2]:
df_decisions = X_test.copy()
df_decisions['PD'] = y_prob
df_decisions['TARGET'] = y_test
df_decisions['RiskBucket'] = pd.cut(
    df_decisions["PD"],
    bins=[0, 0.05, 0.16, 0.45, 1.0],
    labels=["Low", "Medium", "High", "Very High"],
    right=False
)

Risk buckets are defined to align with business decision thresholds,
not model optimization metrics.

In [3]:
df_decisions.groupby("RiskBucket")["TARGET"].mean()

C:\Users\Pratik\AppData\Local\Temp\ipykernel_7140\834082903.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_decisions.groupby("RiskBucket")["TARGET"].mean()


RiskBucket
Low          0.028568
Medium       0.089705
High         0.237534
Very High    0.493151
Name: TARGET, dtype: float64

Observed default rates increase monotonically from Low to High risk,
confirming that calibrated probabilities align with realized outcomes.

| Risk Bucket | PD Range | Suggested Action |
|------------|----------|------------------|
| Low | < 5% | Approve |
| Medium | 5–16% | Approve (Review Terms) |
| High | 30-40% | Manual Review|
| Very High | > 45% | Reject |



Model outputs are mapped to business actions using predefined risk policies.

In [4]:
decision_map = {
    "Low": "Approve",
    "Medium": "Approve (Review Terms)",
    "High": "Manual Review",
    "Very High": "Reject"
}

df_decisions["Decision"] = df_decisions["RiskBucket"].map(decision_map)

While the model’s optimal classification threshold is ~0.65,
business decisions are based on risk buckets to allow graded actions
rather than binary outcomes.

In [5]:
cross_tab = pd.crosstab( df_decisions['RiskBucket'] , y_test, normalize="index")
cross_tab = cross_tab.rename(columns={0: "Non-Default", 1: "Default"})
cross_tab

TARGET,Non-Default,Default
RiskBucket,,
Low,0.971432,0.028568
Medium,0.910295,0.089705
High,0.762466,0.237534
Very High,0.506849,0.493151


Given a policy was assigned to this Risk Bucket, what percentage actually turned out Non-Default vs Default?

#### Model-Level Risk Drivers (SHAP Summary)

Based on SHAP analysis (see Notebook 06), the strongest drivers of default risk in the model are:

- Credit amount and goods price (loan size)
- Credit-to-income related features (repayment burden)
- External credit bureau signals (EXT_SOURCE features)
- Organization-level and income-level historical risk

These features consistently contribute more to the model’s prediction than demographic variables such as age or employment duration.

In [6]:
low_example = df_decisions[df_decisions["RiskBucket"] == "Low"].iloc[0]
med_example = df_decisions[df_decisions["RiskBucket"] == "Medium"].iloc[0]
high_example = df_decisions[df_decisions["RiskBucket"] == "High"].iloc[0]
very_high_example = df_decisions[df_decisions["RiskBucket"] == "Very High"].iloc[0]

In [7]:
cols_to_show = [
    'PD',
    'RiskBucket',
    'Decision',
    'AGE_YEARS',
    'EMPLOYED_YEARS',
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'CREDIT_INCOME_RATIO',
    'INCOME_RISK',
    'ORG_RISK'
]

pd.DataFrame([low_example[cols_to_show]])

,PD,RiskBucket,Decision,AGE_YEARS,EMPLOYED_YEARS,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,CREDIT_INCOME_RATIO,INCOME_RISK,ORG_RISK
191493,0.04119,Low,Approve,48.556164,-13.506849,90000.0,364896.0,19926.0,4.0544,0.086655,0.068697


#### Example 1: Low Risk Applicant

**Model Output**
- Predicted Probability of Default (PD): 4%
- Risk Bucket : Low
- Decision - Approve

**Key Applicant Attributes**
- Age: 48
- Income Risk Score: Low
- Employement Length: Employed for 13.5 years
- Organization Risk: Low
- Amount Credit: High
- Credit-to-Income Ratio: Elevated (mitigated by income and annuity)
- Amount Annuity: Moderate


According to SHAP analysis (Notebook 06), this applicant's low PD is primarily driven by:
- **AMT_INCOME_TOTAL (↓ PD):** Placement in a low-risk income group
- **ORGANIZATION_TYPE risk (↓ PD):** Favorable organizational risk profile
- **AMT_ANNUITY relative to income (↓ PD):** High income combined with a manageable annuity produces a strong negative SHAP contribution, indicating low cash-flow stress
- **AMT_CREDIT / CREDIT_INCOME_RATIO (↑ PD, weak):** Large credit amount contributes mildly toward higher risk, but its impact is outweighed by affordability features
- **Income–Credit interaction (↓ PD):** In the training data, applicants with similar income–credit profiles exhibit lower default frequency
- **OVERALL:** 
The sum of SHAP values is negative, placing the applicant in a low predicted probability of default region

**Decision Rationale:**  
Despite an elevated credit-to-income ratio, the applicant demonstrates strong affordability, income stability, and favorable segment characteristics, resulting in a low modeled probability of default.

In [8]:
pd.DataFrame([med_example[cols_to_show]])

,PD,RiskBucket,Decision,AGE_YEARS,EMPLOYED_YEARS,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,CREDIT_INCOME_RATIO,INCOME_RISK,ORG_RISK
256571,0.051405,Medium,Approve (Review Terms),37.00274,-0.287671,157500.0,770292.0,30676.5,4.890743,0.086655,0.09093


#### Example 2: Medium Risk Applicant

**Model Output**
- Predicted Probability of Default (PD): 5.1%
- Risk Bucket : Medium
- Decision - Approve (Review Terms)

**Key Applicant Attributes**
- Age: 37
- Income Risk Score: Low
- Employement Length: Employed for 3.4 months (short tenure)
- Organization Risk: Low
- Amount Credit: High
- Amount Annuity: Moderate
- Credit Income Ratio: Elevated

According to SHAP analysis (Notebook 06), this applicant's low PD is primarily driven by:
- **AMT_INCOME_TOTAL (↓ PD):** High income places applicant in low-risk income segment, contributing negatively to default risk
- **INCOME_RISK & ORGANIZATION_TYPE risk (↓ PD):** Stable income classification & low-risk employer profile provide favourable SHAP contributions.
- **AMT_ANNUITY relative to income (↓ PD):** Annual debt service represents approximately 19.5% of income, indicating manageable cash-flow stress & producing a negative SHAP contribution.
- **AMT_CREDIT / CREDIT_INCOME_RATIO (↑ PD, weak):** Large credit amount & elevated credit-to-income ratio are primary positive SHAP contributors, increasing modeled risk.\
- **Employement Length (↑ PD, material)** Very short employement tenure, materially increases default risk & is a key differentiator from low-risk approvals.

- **OVERALL:**
Applicant demonstrates strong repayment capacity & favourable income characteristics; however, elevated exposure metrics combined with very short employement tenure increases risk. The net SHAP contributes in a medium risk classification, supporting approval subject to reviewed terms.


**Decision Rationale:**  
Approve is recommended with adjusted terms due to strong affordability & income quality, while mitigating risks related to recent employement & high credit exposure

In [9]:
pd.DataFrame([high_example[cols_to_show]])

,PD,RiskBucket,Decision,AGE_YEARS,EMPLOYED_YEARS,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,CREDIT_INCOME_RATIO,INCOME_RISK,ORG_RISK
103497,0.192564,High,Manual Review,66.30137,-3.263014,148500.0,284400.0,18643.5,1.915152,0.086655,0.093575


#### Example 3: High Risk Applicant

**Model Output**
- Predicted Probability of Default (PD): 19.2%
- Risk Bucket : High
- Decision - Manual Review

**Key Applicant Attributes**
- Age: 66
- Income Risk Score: Low
- Employement Lenght: Employed for 3.26 years
- Organization Risk: Low
- Amount Credit: High
- Amount Annuity: Moderate
- Credit Income Ratio: Moderate

According to SHAP analysis (Notebook 06), this applicant's PD is primarily driven by:
- **AMT_INCOME_TOTAL (↓ PD):** High income places applicant in low-risk income segment, contributing negatively to default risk
- **INCOME_RISK & ORGANIZATION_TYPE risk (↓ PD):** Stable income classification & low-risk employer profile provide favourable SHAP contributions.
- **AGE_YEARS (↑ PD)** Applicant's age significantly increases default risk due to elevated retirement and post-employement income uncertainty.
- **Age X Credit Interaction (↑ PD)** Even with a moderate size loan, expected repayment horizon extends into retirement years, increasing long-term payment risk.
- **Employement Length (↓ PD):** Stable recent employement provides limited risk mitigation but does not offset age-driven risk
- **OVERALL:** 
While affordability & income quality are strong, age-related factors dominate risk profile. The net SHAP contribution results in a high-risk classification, warranting Manual Review.

**Decision Rationale:**
The application is recommended for review due to elevated age-related repayment risk, despite favourable affordability & income characteristics.


In [11]:
pd.DataFrame([very_high_example])

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,...,HAS_CAR,HAS_REALTY,LONG_EMPLOYED,NAME_EDUCATION_ENC,INCOME_RISK,ORG_RISK,PD,TARGET,RiskBucket,Decision
235663,372969,Cash loans,M,0,225000.0,781920.0,61906.5,675000.0,Secondary / secondary special,Married,...,1,1,0.0,1,0.086655,0.092733,0.497291,0,Very High,Reject


#### Example 4 : Very High Risk Applicant

**Model Output**
- Predicted Probability of Default (PD): 49.7%
- Risk Bucket : Very High
- Decision - Reject

**Key Applicant Attributes**
- Age: 50
- Income Risk Score: Low
- Employement Lenght: Employed for 3.7 years
- Organization Risk: Low
- Amount Credit: High
- Amount Annuity: Moderate
- Credit Income Ratio: Elevated

According to SHAP analysis (Notebook 06), the applicant’s elevated PD is primarily driven by high leverage and affordability stress, which outweigh otherwise stable income and employment indicators.

- **CREDIT_INCOME_RATIO (↑ PD):** A ratio of 3.475 indicates that total credit exposure is disproportionately high relative to income. This feature is a dominant positive SHAP contributor reflecting reduced repayment flexibility and higher vulnerability to income shocks.
-**AMT_CREDIT (↑ PD):** The large loan amount materially increases exposure. Higher principal obligations amplify default risk, particularly when combined with already elevated leverage.
-**AMT_ANNUITY (↑ PD):** The annuity imposes a significant recurring repayment burden, compressing disposable income and increasing short-term affordability risk.
-**AGE_YEARS (↑ PD):** While not extreme, age contributes positively to PD by shortening the remaining working horizon, increasing sensitivity to employment or income disruptions during the loan tenure.
-**AMT_INCOME_TOTAL (↓ PD):** Total income of 225K contributes negatively to default risk, reflecting baseline affordability and income capacity.
-**INCOME_RISK & ORGANIZATION_RISK (↓ PD):** Low income risk classification and low-risk employer profile provide stabilizing SHAP contributions, indicating income reliability.
-**EMPLOYMENT_LENGTH (↓ PD):** ontinuous employment over 3.7 years modestly mitigates risk but is insufficient to offset high leverage indicators.
-**Overall:**
Although the applicant demonstrates stable income, employment continuity, and low employer risk, the model identifies excessive credit exposure relative to income as the primary risk driver. The combined effect of a high credit-to-income ratio, large loan amount, and substantial annuity obligation dominates the SHAP profile.
The net SHAP contribution results in a Very High Risk classification, with a predicted default probability approaching 50%.

**Decision Rationale:**
The application is declined due to structural affordability risk and elevated leverage, which materially increase default likelihood despite favourable income stability indicators.